# A8 Calibration & Locked Evaluation (GPU)

Prosedur dua fase yang **immutable** untuk kalibrasi (validation-only) dan
evaluasi locked-test **sekali**. Ikuti `docs/a8-calibration-evaluation-runbook.md`.

**Sebelum mulai:** Runtime > Change runtime type > **T4 GPU**.

**PENTING (one-shot policy):**
1. Phase 1 kalibrasi pada validation SAJA (locked test belum dibaca).
2. **Berhenti** pada *mandatory pause* dan bekukan artifact kalibrasi.
3. Baru setelah konfirmasi manusia, salin locked test dan evaluasi **sekali**.
   Jangan pernah memakai ID evaluasi baru untuk menutupi kegagalan.


## Step 1 — Mount Google Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Konfigurasi path & parameter


In [2]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_SPLIT_DIR = DRIVE_ROOT / "data" / "splits"

# A7 run yang akan dikalibrasi & dievaluasi (hasil notebook 06).
MODEL_RUN_ID = "20260813-1050_indobert-silver-v1"
MODEL_RUN_DIR = DRIVE_ROOT / "runs" / MODEL_RUN_ID

# Direktori split sementara di runtime (Phase 1 hanya manifest + validation).
SPLIT_DIR = Path("/content/a8-splits")

# ID output immutable (jangan pernah diganti setelah percobaan evaluasi).
CALIBRATION_ID = f"{MODEL_RUN_ID}_calibration-v1"
CALIBRATION_DIR = DRIVE_ROOT / "calibration" / CALIBRATION_ID
EVALUATION_ID = f"{MODEL_RUN_ID}_locked-test-v1"
EVALUATION_DIR = DRIVE_ROOT / "evaluation" / EVALUATION_ID
EVIDENCE_ID = f"{MODEL_RUN_ID}_a8-evidence"
EVIDENCE_DIR = DRIVE_ROOT / "evidence" / EVIDENCE_ID

# Baseline metrics dari notebook 05 (keyword + tfidf).
BASELINE_METRICS_DIR = DRIVE_ROOT / "metrics"

PROJECT_DIR = Path("/content/hackathon/ml")

print("Model run dir:", MODEL_RUN_DIR)
print("Drive split dir:", DRIVE_SPLIT_DIR)
print("Runtime split dir:", SPLIT_DIR)
print("Calibration dir:", CALIBRATION_DIR)
print("Evaluation dir:", EVALUATION_DIR)
print("Baseline metrics dir:", BASELINE_METRICS_DIR)


Model run dir: /content/drive/MyDrive/SIPATURE/runs/20260813-1050_indobert-silver-v1
Drive split dir: /content/drive/MyDrive/SIPATURE/data/splits
Runtime split dir: /content/a8-splits
Calibration dir: /content/drive/MyDrive/SIPATURE/calibration/20260813-1050_indobert-silver-v1_calibration-v1
Evaluation dir: /content/drive/MyDrive/SIPATURE/evaluation/20260813-1050_indobert-silver-v1_locked-test-v1
Baseline metrics dir: /content/drive/MyDrive/SIPATURE/metrics


## Step 3 — Verifikasi GPU


In [3]:
!nvidia-smi


Thu Aug 13 16:49:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 4 — Clone repository dari GitHub


In [4]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


Return code: 0

Cloning into '/content/hackathon'...



## Step 5 — Verifikasi commit terbaru (git log)


In [5]:
%cd /content/hackathon/ml
!git log --oneline -3


/content/hackathon/ml
3e90845 (HEAD -> main, origin/main, origin/HEAD) refactor: tidy notebook 07 for re-run (step markers, dynamic run id, remove hardcoded hashes)
bf7dd10 chore: remove stub notebook 07 and renumber calibration notebook 08 -> 07
a898543 docs: record notebook 06 re-run and mark IndoBERT artifacts done


## Step 6 — Uninstall torchvision + install dependencies


In [6]:
%cd /content/hackathon/ml
!python -m pip uninstall -y torchvision
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


/content/hackathon/ml
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**RESTART WAJIB.** Setelah install, restart runtime agar numpy/torch lama tidak
ter-cache di memori.

1. **Runtime > Restart session**
2. Jalankan ulang **Step 1** (mount) dan **Step 2** (config)
3. Step 3–6 **tidak perlu diulang** (repo & packages sudah tersimpan)

Lalu lanjut ke **Step 7**.


## Step 7 — Verifikasi versi package & GPU (setelah restart)


In [3]:
import importlib.util
import torch
import transformers
import numpy

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("NumPy:", numpy.__version__)
print("CUDA tersedia:", torch.cuda.is_available())
print("Torchvision ditemukan:", importlib.util.find_spec("torchvision") is not None)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), "A8 membutuhkan runtime GPU"
assert importlib.util.find_spec("torchvision") is None
print("\nEnvironment A8 siap.")


PyTorch: 2.7.1+cu126
Transformers: 4.53.2
NumPy: 2.2.6
CUDA tersedia: True
Torchvision ditemukan: False
GPU: Tesla T4

Environment A8 siap.


## Step 8 — Import modul sipature_ml


In [4]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"
assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml
print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


Modul SIPATURE berhasil dimuat dari:
/content/hackathon/ml/src/sipature_ml/__init__.py


## Step 9 — Verifikasi model A7 (struktur + hash terhadap manifest)


In [5]:
import json
from pathlib import Path

from sipature_ml.evaluation import validate_model_contract

assert MODEL_RUN_DIR.is_dir(), f"Run A7 tidak ditemukan: {MODEL_RUN_DIR}"

a7_manifest = json.loads(
    (MODEL_RUN_DIR / "manifest.json").read_text(encoding="utf-8")
)
a7_summary = json.loads(
    (MODEL_RUN_DIR / "summary.json").read_text(encoding="utf-8")
)

print("Run ID:", a7_summary.get("run_id"))
print("Status:", a7_summary.get("status"))
print("Test dibaca saat A7:", a7_summary.get("test_read"))

assert a7_summary.get("run_id") == MODEL_RUN_ID
assert a7_summary.get("test_read") is False

verified = validate_model_contract(MODEL_RUN_DIR, a7_manifest)
print("\nModel artifact terverifikasi:", len(verified))
for path in sorted(verified):
    print(f"  {path}")

assert len(verified) == 10
print("\nStruktur & hash model A7 valid.")


Run ID: 20260813-1050_indobert-silver-v1
Status: trained_train_validation_only
Test dibaca saat A7: False

Model artifact terverifikasi: 10
  aspect/model/config.json
  aspect/model/model.safetensors
  aspect/model/special_tokens_map.json
  aspect/model/tokenizer_config.json
  aspect/model/vocab.txt
  polarity/model/config.json
  polarity/model/model.safetensors
  polarity/model/special_tokens_map.json
  polarity/model/tokenizer_config.json
  polarity/model/vocab.txt

Struktur & hash model A7 valid.


## Step 10 — [Phase 1] Siapkan split runtime (manifest + validation saja)


In [6]:
import hashlib
import json
import shutil
from pathlib import Path

MANIFEST_SOURCE = DRIVE_SPLIT_DIR / "split_manifest_silver_v1.json"
VALIDATION_SOURCE = DRIVE_SPLIT_DIR / "validation_silver_v1.jsonl"

assert MANIFEST_SOURCE.is_file()
assert VALIDATION_SOURCE.is_file()

assert not SPLIT_DIR.exists(), (
    f"{SPLIT_DIR} sudah ada. Hapus atau periksa sebelum lanjut."
)
SPLIT_DIR.mkdir(parents=False, exist_ok=False)

shutil.copy2(MANIFEST_SOURCE, SPLIT_DIR / MANIFEST_SOURCE.name)
shutil.copy2(VALIDATION_SOURCE, SPLIT_DIR / VALIDATION_SOURCE.name)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = json.loads((SPLIT_DIR / MANIFEST_SOURCE.name).read_text(encoding="utf-8"))
validation_entry = manifest["outputs"]["validation"]

manifest_sha256 = sha256_file(SPLIT_DIR / MANIFEST_SOURCE.name)
validation_sha256 = sha256_file(SPLIT_DIR / VALIDATION_SOURCE.name)

print("Split version:", manifest["split_version"])
print("Test dikunci:", manifest["test_is_locked"])
print("Manifest SHA-256:", manifest_sha256)
print("Validation SHA-256:", validation_sha256)

assert manifest["test_is_locked"] is True
assert validation_sha256 == validation_entry["sha256"]
assert not (SPLIT_DIR / "test_silver_v1.jsonl").exists()
assert not (SPLIT_DIR / "train_silver_v1.jsonl").exists()

print("\nRuntime Phase 1 hanya berisi manifest + validation.")
print("Locked test TIDAK tersedia di runtime.")


Split version: silver-split-1.0.0
Test dikunci: True
Manifest SHA-256: f77b3777349d2475b1e294e9dd59d7e234a24838c98586e292c45e40419540a8
Validation SHA-256: 7c2f5f911ea33c6854ad1adc21e41a58befb6b24b31cb5ef2b8e03b7b771477c

Runtime Phase 1 hanya berisi manifest + validation.
Locked test TIDAK tersedia di runtime.


## Step 11 — [Phase 1] Jalankan validation-only calibration


In [7]:
import time

import torch

from sipature_ml.evaluation import run_calibration

assert torch.cuda.is_available()
assert not CALIBRATION_DIR.exists(), (
    f"Calibration dir sudah ada: {CALIBRATION_DIR}. Jangan menimpa."
)
assert not (SPLIT_DIR / "test_silver_v1.jsonl").exists()

print("Mulai calibration (validation only)...")
started = time.perf_counter()

calibration = run_calibration(
    split_dir=SPLIT_DIR,
    model_run_dir=MODEL_RUN_DIR,
    output_dir=CALIBRATION_DIR,
)

print("\nCalibration selesai dalam", round(time.perf_counter() - started, 2), "detik")
print("Phase:", calibration["phase"])
print("Test dibaca:", calibration["test_read"])
print("Temperature:", calibration["temperature"])
print("NLL before/after:", calibration["validation_calibration"]["nll_before"],
      "/", calibration["validation_calibration"]["nll_after"])
print("ECE before/after:", calibration["validation_calibration"]["ece_before"],
      "/", calibration["validation_calibration"]["ece_after"])
print("Brier before/after:", calibration["validation_calibration"]["brier_before"],
      "/", calibration["validation_calibration"]["brier_after"])
print("Validation polarity Macro F1:", calibration["validation_polarity"]["macro_f1"])

assert calibration["phase"] == "validation_calibration"
assert calibration["test_read"] is False
assert CALIBRATION_DIR.is_dir()
print("\nVALIDATION CALIBRATION BERHASIL. LOCKED TEST TIDAK DIBACA.")


Mulai calibration (validation only)...

Calibration selesai dalam 29.02 detik
Phase: validation_calibration
Test dibaca: False
Temperature: 0.6
NLL before/after: 0.4533154301304474 / 0.42355896059641346
ECE before/after: 0.2705771093817387 / 0.22529132303198515
Brier before/after: 0.14408784341684366 / 0.13877116723085292
Validation polarity Macro F1: 0.7044036969463576

VALIDATION CALIBRATION BERHASIL. LOCKED TEST TIDAK DIBACA.


## Step 12 — [Phase 1] Verifikasi artifact calibration


In [8]:
import hashlib
import json
from pathlib import Path

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

required = ["calibration.json", "manifest.json", "validation-predictions.npz", "calibration.png"]
for name in required:
    path = CALIBRATION_DIR / name
    print(f"{name}:", "ADA" if path.is_file() else "HILANG")
    assert path.is_file()

frozen_calibration = json.loads(
    (CALIBRATION_DIR / "calibration.json").read_text(encoding="utf-8")
)
calibration_manifest = json.loads(
    (CALIBRATION_DIR / "manifest.json").read_text(encoding="utf-8")
)

calibration_sha256 = sha256_file(CALIBRATION_DIR / "calibration.json")
calibration_manifest_sha256 = sha256_file(CALIBRATION_DIR / "manifest.json")

print("\nCalibration SHA-256:", calibration_sha256)
print("Manifest SHA-256:", calibration_manifest_sha256)
print("Phase:", calibration_manifest["phase"])
print("Test read:", calibration_manifest["test_read"])

assert calibration_sha256 == calibration_manifest["calibration_sha256"]
assert calibration_manifest["phase"] == "validation_calibration"
assert calibration_manifest["test_read"] is False

# Verifikasi seluruh artifact terhadap manifest.
artifact_hashes = calibration_manifest["artifact_hashes"]
for relative, expected in sorted(artifact_hashes.items()):
    actual = sha256_file(CALIBRATION_DIR / relative)
    assert actual == expected, f"Hash tidak cocok: {relative}"

print("\nSeluruh artifact calibration cocok dengan manifest.")


calibration.json: ADA
manifest.json: ADA
validation-predictions.npz: ADA
calibration.png: ADA

Calibration SHA-256: d64f95cdeceb49478d54da0c3eb4c8a053d5bcbc29e0e4d653f5468ad8fc8478
Manifest SHA-256: edeaf063ec93167f5812fec5f26d7ec77e1f42652a4d37b8955961150ccc097a
Phase: validation_calibration
Test read: False

Seluruh artifact calibration cocok dengan manifest.


## Step 13 — [Phase 1] Buat freeze receipt


In [9]:
import json
from datetime import datetime, timezone
from pathlib import Path

FREEZE_RECEIPT = CALIBRATION_DIR.parent / f"{CALIBRATION_ID}_freeze-receipt.json"
assert not FREEZE_RECEIPT.exists(), "Freeze receipt sudah ada."

freeze_receipt = {
    "calibration_id": CALIBRATION_ID,
    "model_run_id": MODEL_RUN_ID,
    "frozen_at": datetime.now(timezone.utc).isoformat(),
    "status": "accepted_and_frozen_before_locked_test",
    "test_read": False,
    "calibration_sha256": calibration_sha256,
    "calibration_manifest_sha256": calibration_manifest_sha256,
    "temperature": frozen_calibration["temperature"],
    "locked_test_policy": (
        "No model, temperature, threshold, taxonomy, or configuration "
        "changes are allowed after this receipt."
    ),
}

FREEZE_RECEIPT.write_text(
    json.dumps(freeze_receipt, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("Freeze receipt:", FREEZE_RECEIPT)
print("Temperature:", frozen_calibration["temperature"])
print("Test dibaca:", freeze_receipt["test_read"])

assert freeze_receipt["test_read"] is False
print("\nKALIBRASI DITERIMA DAN DIBEKUKAN. LOCKED TEST BELUM DIAKSES.")


Freeze receipt: /content/drive/MyDrive/SIPATURE/calibration/20260813-1050_indobert-silver-v1_calibration-v1_freeze-receipt.json
Temperature: 0.6
Test dibaca: False

KALIBRASI DITERIMA DAN DIBEKUKAN. LOCKED TEST BELUM DIAKSES.


## MANDATORY PAUSE

**BERHENTI DI SINI.** Periksa `calibration.json`, `manifest.json`, threshold,
temperature, dan plot di Drive. Bekukan direktori dan jangan ubah apa pun.

Hanya setelah manusia mengonfirmasi artifact diterima, lanjut ke **Step 14**
(masukkan frasa konfirmasi) untuk mengakses locked test.


## Step 14 — Konfirmasi akses locked test (mandatory)


In [10]:
CONFIRMATION_PHRASE = "I AUTHORIZE ONE LOCKED TEST ACCESS"
confirmation = input(f"Ketik persis: {CONFIRMATION_PHRASE}\n")
assert confirmation == CONFIRMATION_PHRASE, "Akses locked test tidak diizinkan"
print("Akses locked test diotorisasi.")


Ketik persis: I AUTHORIZE ONE LOCKED TEST ACCESS
I AUTHORIZE ONE LOCKED TEST ACCESS
Akses locked test diotorisasi.


## Step 15 — [Phase 2] Salin locked test ke runtime split


In [11]:
import hashlib
import json
import shutil
from pathlib import Path

LOCKED_TEST_SOURCE = DRIVE_SPLIT_DIR / "test_silver_v1.jsonl"

assert LOCKED_TEST_SOURCE.is_file(), f"Locked test tidak ditemukan: {LOCKED_TEST_SOURCE}"
assert not (SPLIT_DIR / "test_silver_v1.jsonl").exists()

manifest = json.loads((SPLIT_DIR / "split_manifest_silver_v1.json").read_text(encoding="utf-8"))
test_entry = manifest["outputs"]["test"]

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

assert sha256_file(LOCKED_TEST_SOURCE) == test_entry["sha256"], "Test hash mismatch"

shutil.copy2(LOCKED_TEST_SOURCE, SPLIT_DIR / "test_silver_v1.jsonl")
print("Locked test disalin ke runtime (hash terverifikasi).")
print("Test SHA-256:", test_entry["sha256"])


Locked test disalin ke runtime (hash terverifikasi).
Test SHA-256: edf650024fc2f74c5f3eea1bc04c3b909c52884849067987196fd8b795bb43ff


## Step 16 — [Phase 2] Jalankan locked test evaluation (sekali)


In [12]:
import time

from sipature_ml.evaluation import run_locked_test_evaluation

assert confirmation == CONFIRMATION_PHRASE
assert not EVALUATION_DIR.exists(), (
    f"Evaluation dir sudah ada: {EVALUATION_DIR}. Jangan menimpa."
)

print("Memulai locked test evaluation (one-shot)...")
started = time.perf_counter()

metrics = run_locked_test_evaluation(
    split_dir=SPLIT_DIR,
    model_run_dir=MODEL_RUN_DIR,
    calibration_dir_or_file=CALIBRATION_DIR,
    output_dir=EVALUATION_DIR,
    baseline_metrics_dir=BASELINE_METRICS_DIR,
)

print("\nEvaluasi selesai dalam", round(time.perf_counter() - started, 2), "detik")
print("Phase:", metrics["phase"])
print("Test inference passes:", metrics["test_inference_passes"])
print("Aspect Macro F1:", metrics["aspect"]["macro_f1"])
print("Aspect Micro F1:", metrics["aspect"]["micro_f1"])
print("Polarity Macro F1:", metrics["polarity"]["macro_f1"])
print("ECE:", metrics["calibration"]["ece"])
print("Brier:", metrics["calibration"]["brier"])

assert metrics["test_inference_passes"] == 1
print("\nLOCKED TEST DIEVALUASI TEPAT SEKALI.")


Memulai locked test evaluation (one-shot)...

Evaluasi selesai dalam 12.65 detik
Phase: locked_test_evaluation
Test inference passes: 1
Aspect Macro F1: 0.52472918092202
Aspect Micro F1: 0.5241379310344828
Polarity Macro F1: 0.7459128771731742
ECE: 0.202134303562323
Brier: 0.12582741595904728

LOCKED TEST DIEVALUASI TEPAT SEKALI.


## Step 17 — [Phase 2] Verifikasi hasil evaluation


In [13]:
import hashlib
import json
from pathlib import Path

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

state = json.loads((EVALUATION_DIR / "evaluation-state.json").read_text(encoding="utf-8"))
saved_metrics = json.loads((EVALUATION_DIR / "metrics.json").read_text(encoding="utf-8"))
evaluation_manifest = json.loads((EVALUATION_DIR / "manifest.json").read_text(encoding="utf-8"))

print("State status:", state["status"])
print("Test inference passes:", state["test_inference_passes"])
print("Phase:", evaluation_manifest["phase"])
print("Test SHA-256:", evaluation_manifest["test_sha256"])

assert state["status"] == "completed"
assert state["test_inference_passes"] == 1
assert evaluation_manifest["test_inference_passes"] == 1
assert state["test_sha256"] == evaluation_manifest["test_sha256"]

# Verifikasi seluruh artifact terhadap manifest.
errors = []
for relative, expected in sorted(evaluation_manifest["artifact_hashes"].items()):
    path = EVALUATION_DIR / relative
    if not path.is_file():
        errors.append(f"missing: {relative}")
    elif sha256_file(path) != expected:
        errors.append(f"hash mismatch: {relative}")

print("\nArtifact check:", len(evaluation_manifest["artifact_hashes"]),
      "file,", len(errors), "masalah")
assert not errors

print("\nEVALUASI TELAH SELESAI TEPAT SATU KALI. JANGAN JALANKAN ULANG.")


State status: completed
Test inference passes: 1
Phase: locked_test_evaluation
Test SHA-256: edf650024fc2f74c5f3eea1bc04c3b909c52884849067987196fd8b795bb43ff

Artifact check: 11 file, 0 masalah

EVALUASI TELAH SELESAI TEPAT SATU KALI. JANGAN JALANKAN ULANG.


## Step 18 — [Phase 2] Build safe evidence bundle


In [14]:
import shutil
from pathlib import Path

assert not EVIDENCE_DIR.exists(), "Evidence dir sudah ada."
EVIDENCE_DIR.mkdir(parents=True)

safe_sources = {
    "calibration/calibration.json": CALIBRATION_DIR / "calibration.json",
    "calibration/manifest.json": CALIBRATION_DIR / "manifest.json",
    "calibration/calibration.png": CALIBRATION_DIR / "calibration.png",
    "calibration/freeze-receipt.json": FREEZE_RECEIPT,
    "evaluation/metrics.json": EVALUATION_DIR / "metrics.json",
    "evaluation/manifest.json": EVALUATION_DIR / "manifest.json",
    "evaluation/evaluation-state.json": EVALUATION_DIR / "evaluation-state.json",
    "evaluation/aspect-per-label-f1.png": EVALUATION_DIR / "aspect-per-label-f1.png",
    "evaluation/comparison.png": EVALUATION_DIR / "comparison.png",
    "evaluation/polarity-confusion-matrix.png": EVALUATION_DIR / "polarity-confusion-matrix.png",
    "evaluation/test-probability-quality.png": EVALUATION_DIR / "test-probability-quality.png",
}

for relative, source in safe_sources.items():
    if not source.is_file():
        print(f"(skip, tidak ada) {relative}")
        continue
    destination = EVIDENCE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

print("Safe evidence files disalin:", len(safe_sources))
print("Evidence dir:", EVIDENCE_DIR)


Safe evidence files disalin: 11
Evidence dir: /content/drive/MyDrive/SIPATURE/evidence/20260813-1050_indobert-silver-v1_a8-evidence


## Step 19 — Run summary


In [15]:
import json

print("MODEL RUN ID   :", MODEL_RUN_ID)
print("CALIBRATION ID :", CALIBRATION_ID)
print("EVALUATION ID  :", EVALUATION_ID)
print("EVIDENCE ID    :", EVIDENCE_ID)

print("\nASPECT (locked test):")
print("  Macro F1:", saved_metrics["aspect"]["macro_f1"])
print("  Micro F1:", saved_metrics["aspect"]["micro_f1"])
print("POLARITY:")
print("  Macro F1:", saved_metrics["polarity"]["macro_f1"])
print("  Support:", saved_metrics["polarity"]["support"])
print("PROBABILITY QUALITY:")
print("  ECE:", saved_metrics["calibration"]["ece"])
print("  Brier:", saved_metrics["calibration"]["brier"])
print("SEVERITY:", saved_metrics["severity"]["status"])

print("\nBASELINE COMPARISON:")
for model in saved_metrics["baseline_comparison"]["models"]:
    print(f"  {model['model']}: Macro F1={model['macro_f1']}, Micro F1={model['micro_f1']}")

print("\nDIRS:")
print("  Calibration:", CALIBRATION_DIR)
print("  Evaluation:", EVALUATION_DIR)
print("  Evidence:", EVIDENCE_DIR)

print("\nREMINDER: metrik ini adalah agreement terhadap silver, bukan human-gold.")
print("Jangan retune model/threshold dari hasil locked test.")


MODEL RUN ID   : 20260813-1050_indobert-silver-v1
CALIBRATION ID : 20260813-1050_indobert-silver-v1_calibration-v1
EVALUATION ID  : 20260813-1050_indobert-silver-v1_locked-test-v1
EVIDENCE ID    : 20260813-1050_indobert-silver-v1_a8-evidence

ASPECT (locked test):
  Macro F1: 0.52472918092202
  Micro F1: 0.5241379310344828
POLARITY:
  Macro F1: 0.7459128771731742
  Support: 248
PROBABILITY QUALITY:
  ECE: 0.202134303562323
  Brier: 0.12582741595904728
SEVERITY: unavailable_no_model

BASELINE COMPARISON:
  keyword-silver-v1: Macro F1=0.9767532595118802, Micro F1=0.9783037475345168
  tfidf-aspect-silver-v1: Macro F1=0.7200524598653073, Micro F1=0.804040404040404

DIRS:
  Calibration: /content/drive/MyDrive/SIPATURE/calibration/20260813-1050_indobert-silver-v1_calibration-v1
  Evaluation: /content/drive/MyDrive/SIPATURE/evaluation/20260813-1050_indobert-silver-v1_locked-test-v1
  Evidence: /content/drive/MyDrive/SIPATURE/evidence/20260813-1050_indobert-silver-v1_a8-evidence

REMINDER: met